# 03 — Model Experiments

Inspect the VGG-16 architecture, run a quick training test, and visualize results for the 15-class animal classifier.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.models.vgg16_model import build_vgg16_model, compile_model
from src.models.transfer_learning import TransferLearningManager
from src.models.model_utils import count_parameters, get_model_summary
from src.data.dataset import AnimalDataset
from src.utils.config import load_config
from src.utils.visualization import plot_training_history, plot_confusion_matrix

%matplotlib inline
config = load_config('../config/config.yaml')
CLASS_NAMES = config['data']['classes']

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

## 1. Build and Inspect the 15-Class Model

In [ ]:
model = build_vgg16_model(num_classes=15, freeze_base=True)
model = compile_model(model, learning_rate=0.0001)
model.summary()

In [ ]:
params = count_parameters(model)
print(f'Total parameters:         {params["total"]:>12,}')
print(f'Trainable parameters:     {params["trainable"]:>12,}  ← custom head only (Phase 1)')
print(f'Non-trainable parameters: {params["non_trainable"]:>12,}  ← frozen VGG-16 base')
print(f'Trainable fraction:       {params["trainable"]/params["total"]*100:.1f}%')

## 2. VGG-16 Layer Structure

In [ ]:
vgg_base = None
for layer in model.layers:
    if isinstance(layer, tf.keras.Model):
        vgg_base = layer
        break

if vgg_base:
    print(f'VGG-16 has {len(vgg_base.layers)} layers:\n')
    print(f'{"#":<4} {"Layer Name":<30} {"Type":<25} {"Output Shape":<25} {"Trainable"}')
    print('-' * 95)
    for i, layer in enumerate(vgg_base.layers):
        try:
            out_shape = str(layer.output_shape)
        except:
            out_shape = 'N/A'
        print(f'{i:<4} {layer.name:<30} {type(layer).__name__:<25} {out_shape:<25} {layer.trainable}')

## 3. Test Forward Pass

In [ ]:
dummy_batch = np.random.rand(4, 224, 224, 3).astype(np.float32)
predictions = model.predict(dummy_batch, verbose=0)

print(f'Input shape:  {dummy_batch.shape}')
print(f'Output shape: {predictions.shape}  (should be (4, 15))')
print(f'Output sums:  {predictions.sum(axis=1).round(5)}  (should be ~1.0)')
print(f'\nSample prediction (random weights — not meaningful yet):')
for i, cls in enumerate(CLASS_NAMES):
    bar = '█' * int(predictions[0][i] * 100)
    print(f'  {cls:<12} {predictions[0][i]:.4f}  {bar}')

## 4. Phase 1 vs Phase 2 Parameter Comparison

In [ ]:
tl_manager = TransferLearningManager(config_path='../config/config.yaml')

# Phase 1
model_p1 = tl_manager.build_feature_extraction_model()
params_p1 = count_parameters(model_p1)

# Phase 2
model_p2 = tl_manager.prepare_fine_tuning(model_p1)
params_p2 = count_parameters(model_p2)

print('Parameter comparison:')
print(f'  Phase 1 trainable:  {params_p1["trainable"]:>10,}  (head only)')
print(f'  Phase 2 trainable:  {params_p2["trainable"]:>10,}  (head + last VGG-16 blocks)')
print(f'  Additional params:  {params_p2["trainable"] - params_p1["trainable"]:>10,}')
print(f'  Total params:       {params_p2["total"]:>10,}')

## 5. Quick Training Test (2 epochs)

In [ ]:
from pathlib import Path

if Path('../data/processed/train').exists():
    dataset = AnimalDataset(config_path='../config/config.yaml')
    train_gen, val_gen, _ = dataset.build_generators(augment_train=True)

    test_model = build_vgg16_model(num_classes=15, freeze_base=True)
    test_model = compile_model(test_model, learning_rate=0.0001)

    history = test_model.fit(
        train_gen,
        epochs=2,
        validation_data=val_gen,
        verbose=1,
    )

    print(f'\nTrain accuracy (ep 2):      {history.history["accuracy"][-1]:.4f}')
    print(f'Validation accuracy (ep 2): {history.history["val_accuracy"][-1]:.4f}')
else:
    print('Processed data not found. Run scripts/preprocess_data.py first.')

## 6. Load and Evaluate Trained Model (if available)

In [ ]:
from pathlib import Path
from src.models.model_utils import load_model
from src.training.metrics import compute_metrics, format_metrics_report
from src.utils.visualization import plot_confusion_matrix, plot_sample_predictions

model_path = '../models/final/animal_classifier.h5'

if Path(model_path).exists() and Path('../data/processed/test').exists():
    trained_model = load_model(model_path)
    dataset = AnimalDataset(config_path='../config/config.yaml')
    _, _, test_gen = dataset.build_generators(augment_train=False)

    metrics = compute_metrics(trained_model, test_gen, class_names=CLASS_NAMES)
    print(format_metrics_report(metrics, class_names=CLASS_NAMES))

    plot_confusion_matrix(
        metrics['confusion_matrix'],
        class_names=CLASS_NAMES,
        save_path='../outputs/figures/confusion_matrix_notebook.png'
    )

    # Sample predictions
    test_gen.reset()
    imgs, lbls = next(iter(test_gen))
    y_pred = np.argmax(trained_model.predict(imgs, verbose=0), axis=1)
    y_true = np.argmax(lbls, axis=1)
    plot_sample_predictions(
        imgs, y_true, y_pred,
        class_names=CLASS_NAMES,
        save_path='../outputs/figures/sample_predictions_notebook.png'
    )
else:
    print('Trained model or test data not found.')
    print('Run: python scripts/train.py')